In [1]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path("datasets/constructed_datasets")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REGISTRY_OUTPUT_FILE = OUTPUT_DIR / "registry_reference_cohort.csv"
CHIA_OUTPUT_FILE = OUTPUT_DIR / "chia_trials.csv"
LCT_OUTPUT_FILE = OUTPUT_DIR / "lct_trials.csv"

registry_df = pd.read_csv("ctg_full_2005_2025/cohort_qc/analytic_cohort.csv")
chia_df = pd.read_csv("benchmark_datasets/chia_ctg_retrieval.csv")
lct_df = pd.read_csv("benchmark_datasets/lct_ctg_retrieval.csv")

print("Registry:", registry_df.shape)
print("CHIA:", chia_df.shape)
print("LCT:", lct_df.shape)

print("\nRegistry columns:")
print(registry_df.columns.tolist())

print("\nCHIA columns:")
print(chia_df.columns.tolist())

print("\nLCT columns:")
print(lct_df.columns.tolist())

Registry: (405875, 27)
CHIA: (1000, 8)
LCT: (1006, 8)

Registry columns:
['nct_id', 'source_page', 'study_type', 'study_type_expected', 'study_status', 'study_start_date', 'study_start_year', 'start_date_precision', 'start_year_in_range', 'first_posted_date', 'first_posted_year', 'phase', 'conditions', 'condition_meshes', 'condition_ancestors', 'lead_sponsor', 'sponsor_class', 'enrollment', 'sex', 'minimum_age', 'maximum_age', 'countries', 'n_countries', 'locations', 'eligibility_available', 'eligibility_char_count', 'eligibility_text']

CHIA columns:
['dataset', 'nct_id', 'resolved_nct_id', 'id_remapped', 'request_success', 'api_error', 'eligibility_text', 'retrieved_at']

LCT columns:
['dataset', 'nct_id', 'resolved_nct_id', 'id_remapped', 'request_success', 'api_error', 'eligibility_text', 'retrieved_at']


In [2]:
print("\nCHIA CHECKS")
print(f"Rows: {len(chia_df)}")
print(f"Unique NCT IDs: {chia_df['nct_id'].nunique()}")
print(f"Missing NCT IDs: {chia_df['nct_id'].isna().sum()}")
print(f"Duplicate NCT IDs: {chia_df['nct_id'].duplicated().sum()}")
print(f"Missing eligibility text: {chia_df['eligibility_text'].isna().sum()}")

print("\nLCT CHECKS")
print(f"Rows: {len(lct_df)}")
print(f"Unique NCT IDs: {lct_df['nct_id'].nunique()}")
print(f"Missing NCT IDs: {lct_df['nct_id'].isna().sum()}")
print(f"Duplicate NCT IDs: {lct_df['nct_id'].duplicated().sum()}")
print(f"Missing eligibility text: {lct_df['eligibility_text'].isna().sum()}")


CHIA CHECKS
Rows: 1000
Unique NCT IDs: 1000
Missing NCT IDs: 0
Duplicate NCT IDs: 0
Missing eligibility text: 0

LCT CHECKS
Rows: 1006
Unique NCT IDs: 1006
Missing NCT IDs: 0
Duplicate NCT IDs: 0
Missing eligibility text: 0


In [3]:
chia_ids = set(chia_df["nct_id"])
lct_ids = set(lct_df["nct_id"])

chia_lct_overlap = chia_ids & lct_ids
benchmark_union = chia_ids | lct_ids

print("\nBENCHMARK OVERLAP")
print(f"CHIA NCT IDs: {len(chia_ids)}")
print(f"LCT NCT IDs: {len(lct_ids)}")
print(f"CHIA-LCT overlap: {len(chia_lct_overlap)}")
print(f"Unique NCT IDs across benchmarks: {len(benchmark_union)}")

if chia_lct_overlap:
    print("\nOverlapping NCT IDs:")
    print(sorted(chia_lct_overlap))


BENCHMARK OVERLAP
CHIA NCT IDs: 1000
LCT NCT IDs: 1006
CHIA-LCT overlap: 0
Unique NCT IDs across benchmarks: 2006


In [4]:
registry_ids = set(registry_df["nct_id"])

chia_in_registry = chia_ids & registry_ids
lct_in_registry = lct_ids & registry_ids
benchmark_in_registry = benchmark_union & registry_ids

print("\nREGISTRY-BENCHMARK OVERLAP")

print(f"Registry NCT IDs: {len(registry_ids)}")

print(
    f"CHIA trials present in registry: "
    f"{len(chia_in_registry)} / {len(chia_ids)}"
)

print(
    f"LCT trials present in registry: "
    f"{len(lct_in_registry)} / {len(lct_ids)}"
)

print(
    f"Unique benchmark trials present in registry: "
    f"{len(benchmark_in_registry)}"
)

print(
    f"CHIA trials outside registry cohort: "
    f"{len(chia_ids - registry_ids)}"
)

print(
    f"LCT trials outside registry cohort: "
    f"{len(lct_ids - registry_ids)}"
)


REGISTRY-BENCHMARK OVERLAP
Registry NCT IDs: 405875
CHIA trials present in registry: 952 / 1000
LCT trials present in registry: 687 / 1006
Unique benchmark trials present in registry: 1639
CHIA trials outside registry cohort: 48
LCT trials outside registry cohort: 319


In [5]:
registry_reference_df = registry_df.loc[
    ~registry_df["nct_id"].isin(benchmark_union)
].copy()

print("\nREGISTRY REFERENCE COHORT")

print(f"Registry before benchmark exclusion: {len(registry_df)}")
print(f"Benchmark trials removed: {len(registry_df) - len(registry_reference_df)}")
print(f"Registry after benchmark exclusion: {len(registry_reference_df)}")

print(
    f"Unique NCT IDs after exclusion: "
    f"{registry_reference_df['nct_id'].nunique()}"
)

print(
    f"Duplicate NCT IDs after exclusion: "
    f"{registry_reference_df['nct_id'].duplicated().sum()}"
)

remaining_benchmark_ids = (
    set(registry_reference_df["nct_id"])
    & benchmark_union
)

print(
    f"Benchmark NCT IDs remaining in registry: "
    f"{len(remaining_benchmark_ids)}"
)


REGISTRY REFERENCE COHORT
Registry before benchmark exclusion: 405875
Benchmark trials removed: 1639
Registry after benchmark exclusion: 404236
Unique NCT IDs after exclusion: 404236
Duplicate NCT IDs after exclusion: 0
Benchmark NCT IDs remaining in registry: 0


In [6]:
expected_registry_size = (
    len(registry_df)
    - len(benchmark_in_registry)
)

assert len(registry_reference_df) == expected_registry_size

assert not (
    set(registry_reference_df["nct_id"])
    & benchmark_union
)

print("Registry-benchmark exclusion reconciled successfully.")

Registry-benchmark exclusion reconciled successfully.


In [7]:
registry_reference_df.to_csv(
    REGISTRY_OUTPUT_FILE,
    index=False
)

chia_df.to_csv(
    CHIA_OUTPUT_FILE,
    index=False
)


lct_df.to_csv(
    LCT_OUTPUT_FILE,
    index=False
)

print(f"Saved: {REGISTRY_OUTPUT_FILE}")
print(f"Saved: {CHIA_OUTPUT_FILE}")
print(f"Saved: {LCT_OUTPUT_FILE}")

Saved: datasets/constructed_datasets/registry_reference_cohort.csv
Saved: datasets/constructed_datasets/chia_trials.csv
Saved: datasets/constructed_datasets/lct_trials.csv
